# Exp13 · variasi sensor sebelum soal
Bandingkan median episode exp11 dengan normalisasi variasi peserta. Gunakan paket v3, dua CSV v2 15 detik dan rating v4 seperti exp11. Jalankan semua sel pada CPU.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
DATASET_ROOT = PROJECT_ROOT / 'Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-06/v3/exp13/outputs'

## Validasi paket, fitur sensor dan metrik

In [ ]:
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import RobustScaler

HRV = ["rmssd_ms", "sdnn_ms"]
EYE = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean",
       "ear_right_std", "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean"]
CORE = HRV + EYE
ID = ["participant_id", "episode_id"]
SEED = 42


def read_bundle(root, case):
    proxy = case == "v3"
    package = root / ("v3_behavioral_proxy" if proxy else "v4_self_reported_rating")
    target = "cognitive_load_label" if proxy else "difficulty_rating"
    labels = ["Low", "Medium", "High"] if proxy else list(range(1, 11))
    window_dir = Path("window_15s_step_5s")
    hashes = {}

    def read(relative, columns):
        path = package / relative
        hashes[str(Path(relative).as_posix())] = hashlib.sha256(path.read_bytes()).hexdigest()
        return pd.read_csv(path, usecols=columns)

    def attach(left, right, keys, validation):
        merged = left.merge(right, on=keys, how="left", validate=validation, indicator=True)
        if not merged.pop("_merge").eq("both").all():
            raise ValueError(f"CSV pendamping tidak cocok pada {keys}.")
        return merged

    keys = ["window_id", "participant_id", "session_id", target]
    data = read(window_dir / "good_acceptable.csv", keys + CORE)
    assignments = read(window_dir / "window_assignments.csv",
                       keys + ["episode_id"] + (["split"] if proxy else []))
    for table in (data, assignments):
        if table.window_id.duplicated().any() or table[keys].isna().any().any():
            raise ValueError("Kunci window tidak lengkap atau window_id berulang.")
    data = attach(data, assignments, keys, "one_to_one")
    evidence = read("episode_labels.csv" if proxy else "episode_ratings.csv",
                    ID + ["session_id", target])
    if evidence.episode_id.isna().any() or evidence.episode_id.duplicated().any():
        raise ValueError("episode_id pada bukti harus unik dan lengkap.")
    data = attach(data, evidence, ID + ["session_id", target], "many_to_one")
    if set(data[target]) != set(labels) or not np.isfinite(data[CORE].to_numpy(float)).all():
        raise ValueError("Periksa kelas target dan kelengkapan sepuluh fitur core.")
    if proxy:
        splits = read("participant_splits.csv", ["participant_id", "split"])
        if splits.participant_id.duplicated().any() or set(splits.split) != {"Train", "validation", "Test"}:
            raise ValueError("Periksa pembagian peserta v3.")
        data = attach(data, splits, ["participant_id", "split"], "many_to_one")
        weights = read(window_dir / "train_sample_weights.csv", ["window_id", *ID, target, "sample_weight"])
        train = data.split.eq("Train")
        if set(weights.window_id) != set(data.loc[train, "window_id"]) or weights.window_id.duplicated().any():
            raise ValueError("Daftar bobot berbeda dari window Train.")
        checked = attach(data.loc[train], weights, ["window_id", *ID, target], "one_to_one")
        if not np.isfinite(checked.sample_weight).all() or not checked.sample_weight.gt(0).all():
            raise ValueError("Bobot Train harus positif dan finite.")
        data["sample_weight"] = data.window_id.map(checked.set_index("window_id").sample_weight)
    return data.rename(columns={target: "actual"}), labels, hashes

def scores(table, labels):
    actual, predicted = table.actual, table.predicted
    row = dict(n=len(table), participants=table.participant_id.nunique(),
               accuracy=accuracy_score(actual, predicted),
               macro_f1=f1_score(actual, predicted, labels=labels, average="macro", zero_division=0))
    if len(labels) == 10:
        distance = (actual - predicted).abs()
        row.update(mae=distance.mean(), within_one=distance.le(1).mean())
    return row

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

## Perkirakan variasi dari window sebelum soal dan uji kedua model per peserta

In [ ]:
import itertools
from bisect import bisect_right
from collections import defaultdict
LEGACY_POLICY = 'behavioral_concordance_v1'
POLICY = 'behavioral_concordance_v2'
MIN_REFERENCE = 5

def make_reference(episodes, splits, *, policy=POLICY):
    """One first-exposure observation per person/item; only successful Train cases."""
    seen, items = set(), defaultdict(list)
    for row in sorted(episodes, key=lambda r: (r['start_ms'], r['episode_id'])):
        identity = row['participant_id'], row['item_key']
        if identity in seen:
            continue
        seen.add(identity)
        if (splits[row['participant_id']] == 'Train' and row['valid'] and row['completed']
                and (policy == LEGACY_POLICY or row.get('reference_eligible', True))
                and row['success_submission_type'] == 'manual' and row['active_seconds'] > 0):
            items[row['item_key']].append({'participant_id': row['participant_id'],
                'episode_id': row['episode_id'], 'active_seconds': row['active_seconds']})
    return {'policy': policy, 'minimum_other_people': MIN_REFERENCE, 'quantiles': [1/3,2/3],
            'quantile_method': 'linear', 'source_split': 'Train', 'self_reference': False,
            'items': {key: sorted(values, key=lambda v: v['participant_id']) for key, values in sorted(items.items())}}

def reference_for(reference, episode):
    values = [v for v in reference['items'].get(episode['item_key'], [])
              if v['participant_id'] != episode['participant_id']]
    result = {'participants': [v['participant_id'] for v in values], 'count': len(values),
                  'lower_seconds': None, 'upper_seconds': None, 'reason': ''}
    if len(values) < MIN_REFERENCE:
        result['reason'] = 'insufficient_reference'
    else:
        lower, upper = np.quantile([v['active_seconds'] for v in values], [1/3,2/3], method='linear')
        result.update(lower_seconds=float(lower), upper_seconds=float(upper))
        if not 0 < lower < upper:
            result['reason'] = 'degenerate_reference'
    return result

def label_episode(episode, reference):
    limits = reference_for(reference, episode)
    result = {'cognitive_load_label': '', 'label_reason': '', 'time_band': '', 'performance_band': '',
                  'reference_count': limits['count'], 'reference_participants': limits['participants'],
                  'lower_seconds': limits['lower_seconds'], 'upper_seconds': limits['upper_seconds']}
    if not episode['valid']:
        result['label_reason'] = 'invalid_event_trace'
    elif limits['reason']:
        result['label_reason'] = limits['reason']
    else:
        seconds, errors = episode['active_seconds'], episode['manual_errors']
        time_band = ('Low' if seconds <= limits['lower_seconds'] else
                     'Medium' if seconds <= limits['upper_seconds'] else 'High')
        performance = ('Low' if episode['completed'] and errors == 0 and episode['manual_submissions'] == 1
                       and episode['success_submission_type'] == 'manual' else
                       'Medium' if episode['completed'] and 1 <= errors <= 2 else 'High' if errors >= 3 else '')
        result.update(time_band=time_band, performance_band=performance)
        # An unfinished duration is only a lower bound: never use it for Low/Medium.
        if performance == time_band and (episode['completed'] or time_band == 'High'):
            result.update(cognitive_load_label=time_band, label_reason='concordant_behavioral_evidence')
        else:
            result['label_reason'] = 'conflicting_or_insufficient_behavior'
    return result

class WindowIndex:
    """Match only continuous active time from one episode, without rounding gaps."""

    def __init__(self, episodes):
        self.sessions = defaultdict(list)
        for episode in episodes:
            merged = []
            for start, end in sorted(episode['intervals']):
                if end <= start:
                    raise ValueError('Invalid active interval.')
                if merged and start <= merged[-1][1]:
                    merged[-1][1] = max(end, merged[-1][1])
                else:
                    merged.append([start,end])
            self.sessions[episode['session_id']].extend((start,end,episode) for start,end in merged)
        self.starts = {}
        for session, intervals in self.sessions.items():
            intervals.sort(key=lambda r: r[0])
            if any(b[0] < a[1] for a,b in itertools.pairwise(intervals)):
                raise ValueError('Overlapping active episodes.')
            self.starts[session] = [r[0] for r in intervals]

    def match(self, session_id, start, end):
        if end <= start:
            raise ValueError('Invalid window interval.')
        i = bisect_right(self.starts.get(session_id, []), start)-1
        if i >= 0:
            _, stop, episode = self.sessions[session_id][i]
            if end <= stop:
                return episode, ''
        return None, 'not_single_active_episode'

def epoch_ms(value):
    timestamp = pd.Timestamp(value)
    if pd.isna(timestamp) or timestamp.tzinfo is None:
        raise ValueError(f"Expected a timezone-aware timestamp: {value!r}")
    return timestamp.timestamp() * 1000

HEAD = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
SENSORS = CORE + HEAD

def history_streams(path):
    keys = ["participant_id", "session_id"]
    bank = pd.read_csv(path, usecols=keys + ["window_start", "window_end"] + SENSORS)
    for key in ["window_start", "window_end"]:
        bank[key] = pd.to_datetime(bank[key], utc=True, format="mixed", errors="raise")
    if bank[keys + ["window_start", "window_end"]].isna().any().any() or not bank.window_end.gt(bank.window_start).all():
        raise ValueError("Interval sensor historis tidak valid.")
    bank = bank[np.isfinite(bank[SENSORS]).all(axis=1)]
    return dict(tuple(bank.sort_values("window_end", ascending=False).groupby(keys)))

def prior_reference(stream, start):
    # Newest first. Work only with disjoint windows that finished before this task.
    chosen = []
    limit = start
    if stream is not None:
        for index, left, right in stream[["window_start", "window_end"]].itertuples():
            if right <= limit:
                chosen.append(index)
                limit = left
                if len(chosen) == 12:
                    break
    if len(chosen) < 3:
        return pd.Series(np.nan, index=SENSORS), len(chosen), None
    return stream.loc[chosen, SENSORS].median(), len(chosen), stream.at[chosen[0], "window_end"]

"""Standalone notebook cells for participant-held-out evaluation of the v3 target."""

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

THREE = ["Low", "Medium", "High"]
TWO = ["Non-High", "High"]


def binary(y):
    return y.where(y.eq("High"), "Non-High")


def snapshot_hash(path, expected_lf_hash):
    data = path.read_bytes()
    if hashlib.sha256(data.replace(b"\r\n", b"\n")).hexdigest() != expected_lf_hash:
        raise ValueError(f"Gunakan snapshot v2 15 detik dari repo untuk exp07: {path.name}")
    return hashlib.sha256(data).hexdigest()


def build_pool(bank, all_windows, records, official):
    # Use all quality-eligible sensor windows, including episodes unlabeled by the full-Train reference.
    streams = history_streams(bank)
    frame = pd.read_csv(all_windows, usecols=["window_id", "session_id", "participant_id", "window_start",
                                             "window_end", "window_phase_mix", "quality_tier"] + SENSORS)
    frame = frame[frame.quality_tier.isin(["good", "acceptable"])]
    matcher = WindowIndex(records)
    matches = []
    for row in frame.itertuples():
        episode, _ = matcher.match(row.session_id, epoch_ms(row.window_start), epoch_ms(row.window_end))
        active = all(p["study_phase"] == "active_task" for p in json.loads(row.window_phase_mix)
                     if p["duration_seconds"] > 0)
        if episode and active:
            if episode["participant_id"] != row.participant_id:
                raise ValueError("Pemetaan peserta berbeda.")
            matches.append((row.Index, episode["episode_id"]))
    mapped = frame.loc[[i for i, _ in matches]].copy()
    mapped["episode_id"] = [e for _, e in matches]
    known = pd.DataFrame(records).set_index("episode_id")
    published = mapped[mapped.episode_id.map(known.cognitive_load_label).ne("")]
    cols = ["window_id", "participant_id", "episode_id"]
    pd.testing.assert_frame_equal(published[cols].sort_values(cols).reset_index(drop=True),
                                  official[cols].sort_values(cols).reset_index(drop=True), check_dtype=False)
    # No Test feature vectors enter model development.
    dev = mapped[mapped.episode_id.map(known.split).isin(["Train", "validation"])]
    if not np.isfinite(dev[SENSORS]).all().all():
        raise ValueError("Fitur sensor tidak lengkap.")
    x = dev.groupby("episode_id")[SENSORS].median()
    meta = known.loc[x.index, ["participant_id", "session_id", "start_ms", "split"]]
    refs = []
    for row in meta.itertuples():
        stream = streams.get((row.participant_id, row.session_id))
        start = pd.to_datetime(row.start_ms, unit="ms", utc=True)
        ref, _, _ = prior_reference(stream, start)
        refs.append(ref)
    return x, pd.DataFrame(refs, index=x.index), meta


def fold_labels(records, fit_people):
    roles = {row["participant_id"]: ("Train" if row["participant_id"] in fit_people else "held")
             for row in records}
    reference = make_reference(records, roles)
    return pd.Series({r["episode_id"]: label_episode(r, reference)["cognitive_load_label"]
                      for r in records})


"""Source cells following the exp07 data-loading functions in the standalone notebook."""

from functools import lru_cache

THRESHOLDS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0)
MIN_RECALL = 0.75
BLOCKS = {"HRV": HRV, "Eye": EYE, "Head": HEAD}


def load_cohort(root):
    official, _, hashes = read_bundle(root, "v3")
    package = root / "v3_behavioral_proxy"
    ev = pd.read_csv(package / "episode_labels.csv", keep_default_na=False, float_precision="round_trip")
    ev["intervals"] = ev.intervals.map(json.loads)
    records = ev.to_dict("records")
    people = pd.read_csv(package / "participant_splits.csv")
    train = frozenset(people.loc[people.split.eq("Train"), "participant_id"])
    val = frozenset(people.loc[people.split.eq("validation"), "participant_id"])
    if not ev.episode_id.map(fold_labels(records, train)).eq(ev.cognitive_load_label).all():
        raise ValueError("Label split resmi berbeda.")
    wd = root / "v2_session_phase/window_15s_step_5s"
    bank, pool = wd / "good_acceptable.csv", wd / "all_data.csv"
    hashes["v2_history"] = snapshot_hash(bank, "39e72fd8504bb57d83e2edd91a371e904ba6203ad5a3df90fa560a3eaa715722")
    hashes["v2_all"] = snapshot_hash(pool, "acbef9a60f4114de460ee3c57ca94ff5cae0d48b67229d8b14283d173fef1355")
    x, refs, meta = build_pool(bank, pool, records, official)
    return x, refs, meta, records, train, val, bank, hashes


def high_score(model, x):
    return model.decision_function(x) * (1 if model.classes_[1] == "High" else -1)

def hinge(y, margin):
    return np.maximum(0, 1 - np.where(y.eq("High"), 1, -1) * margin)

def evaluate(predictions):
    summary, paired = [], []
    for (protocol, recipe), g in predictions.groupby(["protocol", "recipe"]):
        if g.episode_id.duplicated().any():
            raise ValueError("Prediksi episode berulang.")
        actual = g.actual.eq("High")
        aucs, counts = [], []
        for participant, part in g.groupby("participant_id"):
            high = part.actual.eq("High")
            n_pairs = int(high.sum() * (~high).sum())
            if n_pairs:
                auc = roc_auc_score(high, part.score)
                aucs.append(auc)
                counts.append(n_pairs)
                paired.append(dict(protocol=protocol, recipe=recipe, participant_id=participant,
                                   pairs=n_pairs, concordance=auc))
        summary.append(dict(protocol=protocol, recipe=recipe, episodes=len(g),
                            accuracy=g.actual.eq(g.predicted).mean(),
                            balanced_accuracy=balanced_accuracy_score(g.actual, g.predicted),
                            macro_f1=f1_score(g.actual, g.predicted, average="macro"),
                            auc_pooled=roc_auc_score(actual, g.score), hinge=hinge(g.actual, g.score).mean(),
                            paired_people=len(aucs), within_person_pairs=sum(counts),
                            concordance_person_mean=np.mean(aucs) if aucs else np.nan,
                            concordance_pair_mean=np.average(aucs, weights=counts) if aucs else np.nan))
    return pd.DataFrame(summary), pd.DataFrame(paired)

def compare(predictions, ratings):
    rows = []
    for (protocol, model), group in predictions.groupby(["protocol", "model"]):
        data = group.merge(ratings, on=["episode_id", "participant_id"], how="left", validate="one_to_one")
        rated = data.dropna(subset=["difficulty_rating"])
        high = rated.difficulty_rating.ge(7)
        rho = (spearmanr(rated.score, rated.difficulty_rating).statistic
               if rated.score.nunique() > 1 and rated.difficulty_rating.nunique() > 1 else None)
        rows.append(dict(protocol=protocol, model=model, eligible=len(group), matched=len(rated),
                         participants=rated.participant_id.nunique(), rating_7_10=high.sum(),
                         spearman=rho, auc_rating_7_10=roc_auc_score(high, rated.score) if high.nunique() == 2 else None))
    return pd.DataFrame(rows)

"""Source cells for the controlled episode/window SVM comparison."""

from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score, roc_auc_score


"""Paired evaluation of personal pre-episode IQR scaling for the v3 behavioral proxy."""

MIN_IQR_WINDOWS = 4


def historical_iqr(bank, meta, references):
    streams = history_streams(bank)
    spreads = pd.DataFrame(np.nan, index=meta.index, columns=SENSORS)
    audit = []
    for episode, row in meta.iterrows():
        start = pd.to_datetime(row.start_ms, unit="ms", utc=True)
        stream = streams.get((row.participant_id, row.session_id))
        selected = []
        limit = start
        if stream is not None:
            for idx, left, right in stream[["window_start", "window_end"]].itertuples():
                if right <= limit:
                    selected.append(idx)
                    limit = left
                    if len(selected) == 12:
                        break
        last = pd.NaT
        if selected:
            last = stream.at[selected[0], "window_end"]
        if len(selected) >= 3:
            history = stream.loc[selected, SENSORS]
            np.testing.assert_allclose(history.median(), references.loc[episode], rtol=0, atol=0)
            if len(history) >= MIN_IQR_WINDOWS:
                spreads.loc[episode] = history.quantile(0.75) - history.quantile(0.25)
        elif references.loc[episode].notna().any():
            raise ValueError("Referensi pusat tidak sesuai dengan riwayat.")
        audit.append(dict(episode_id=episode, history_windows=len(selected), last_history_end=last,
                          episode_start=start))
    return spreads, pd.DataFrame(audit).set_index("episode_id")


def evaluate_fold(x, refs, meta, labels, spread, fitting, held):
    if fitting & held:
        raise ValueError("Peserta fitting dan evaluasi tumpang tindih.")
    eligible = labels.ne("")
    fit = eligible & meta.participant_id.isin(fitting)
    test = eligible & meta.participant_id.isin(held)
    if not test.any():
        return [], []
    y = binary(labels[fit])
    people = meta.loc[fit, "participant_id"]
    weights = 1 / (people.groupby(y).transform("nunique") * y.groupby([y, people]).transform("size"))
    weights /= weights.mean()
    delta = x - refs.fillna(x.loc[fit].median())
    # Give each fitting participant one vote per feature for the typical historical IQR.
    by_person = spread.loc[fit].groupby(people).median()
    typical = by_person.median()
    # Average local and population IQR to moderate short-history estimates.
    # A zero population IQR or missing history leaves the exp11 feature unchanged.
    ratio = spread.div(typical.where(typical.gt(0)))
    multiplier = (2 / (1 + ratio)).fillna(1.0)
    results = []
    for name, features in [("exp11_center", delta), ("personal_iqr", delta * multiplier)]:
        scaler = StandardScaler().fit(features.loc[fit], sample_weight=weights)
        model = SVC(C=1, gamma="scale", kernel="rbf")
        model.fit(scaler.transform(features.loc[fit]), y, sample_weight=weights)
        if model.fit_status_ != 0:
            raise ValueError("SVM belum konvergen.")
        scores = high_score(model, scaler.transform(features.loc[test]))
        results.append(meta.loc[test, ["participant_id"]].assign(original=labels[test], actual=binary(labels[test]),
                       recipe=name, score=scores, predicted=np.where(scores > 0, "High", "Non-High"),
                       personal_scale_features=ratio.loc[test].count(axis=1)).reset_index())
    audit = pd.DataFrame({"typical_iqr": typical, "fit_people_with_iqr": by_person.count()}).rename_axis("feature")
    return results, audit.reset_index()


def run(root, output_root):
    x, refs, meta, records, train, val, bank, hashes = load_cohort(root)
    spread, history = historical_iqr(bank, meta, refs)
    splits = [("Train_LOPO", p, train - {p}, {p}) for p in sorted(train)]
    splits.append(("validation", "validation", train, val))
    results, audits = [], []
    for protocol, fold, fitting, held in splits:
        labels = fold_labels(records, fitting).reindex(x.index)
        rows, scales = evaluate_fold(x, refs, meta, labels, spread, fitting, held)
        if rows:
            results.extend(row.assign(protocol=protocol, fold=fold) for row in rows)
            audits.append(scales.assign(protocol=protocol, fold=fold))
    pred = pd.concat(results, ignore_index=True).join(history, on="episode_id", validate="many_to_one")
    metrics, pairs = evaluate(pred)
    counts = []
    for (protocol, recipe), g in pred.groupby(["protocol", "recipe"]):
        tn, fp, fn, tp = confusion_matrix(g.actual, g.predicted, labels=TWO).ravel()
        counts.append(dict(protocol=protocol, recipe=recipe, tn=tn, fp=fp, fn=fn, tp=tp,
                           people=g.participant_id.nunique(), episodes_with_scale=g.personal_scale_features.gt(0).sum()))
    metrics = metrics.merge(pd.DataFrame(counts), on=["protocol", "recipe"], validate="one_to_one")
    rating_file = root / "v4_self_reported_rating/episode_ratings.csv"
    ratings = pd.read_csv(rating_file, usecols=["episode_id", "participant_id", "difficulty_rating"])
    alignment = compare(pred.rename(columns={"recipe": "model"}), ratings)
    out = output_root / ("iqr_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    for name, frame in [("episode_predictions", pred), ("metrics", metrics), ("within_person", pairs),
                        ("fold_iqr", pd.concat(audits)), ("rating_alignment", alignment)]:
        frame.to_csv(out / (name + ".csv"), index=False)
    hashes["v4_episode_ratings"] = hashlib.sha256(rating_file.read_bytes()).hexdigest()
    save_json(out / "manifest.json", {"inputs": hashes, "features": SENSORS, "C": 1, "gamma": "scale",
              "threshold": 0, "minimum_iqr_windows": MIN_IQR_WINDOWS, "maximum_history_windows": 12,
              "multiplier": "2 / (1 + local_IQR / typical_fitting_IQR), fallback 1", "test_evaluated": False,
              "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__})
    print(metrics[["protocol", "recipe", "episodes", "accuracy", "auc_pooled", "tn", "fp", "fn", "tp",
                   "episodes_with_scale"]].to_string(index=False))
    print(alignment.to_string(index=False))
    archive = shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name)
    print("ZIP:", archive)
    return Path(archive)


## Evaluasi, audit ketersediaan riwayat, dan simpan ZIP

In [ ]:
archive = run(DATASET_ROOT, OUTPUT_ROOT)